# Coster warehouse data: what we have and what it tells us

Coster runs an automated warehouse and shared nine Excel files with us on 23 Oct 2026.
This notebook works through them in three steps: first we clean the files, then we explore them, and only then do we analyse.

**What we want to understand**

1. How occupancy develops over time, per cell class.
2. The life of a load unit: why it comes into the warehouse, and when and why it leaves.
3. Whether the warehouse follows its own production plan.
4. What each date in the files means.
5. The capacity rules: which load unit type may go into which cell type.

For each question the notebook ends with a plain answer and a status: **answered**, **partly** or **not in the data yet**.
Where the data cannot answer, we write down what we need to ask Coster.

**How to read it.** Each analysis step starts with the question, shows one chart or small table, and ends with the answer.
Every number in the text is computed by the code cell that prints it.

**Contents**

0. Setup
1. The data Coster gave us
2. Cleaning
3. Exploration
4. Analysis (4.1 to 4.5, one per question)
5. Summary and questions for Coster

## 0. Setup

A **load unit (UDC, *unità di carico*)** is the thing one warehouse cell holds: a pallet, a pallbox or a container.
From here on we write "load unit".

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:,.1f}".format)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.color": "#e6e6e6",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.axisbelow": True,
    }
)

DATA = Path(os.environ.get("COSTER_DATA_DIR", "../../data"))  # outside the repo
OUT = Path("../outputs/audit")  # git-ignored
OUT.mkdir(parents=True, exist_ok=True)
FILE_DATE = pd.Timestamp("2026-10-23")  # date in the file names


def say(text):
    """Show a sentence built from computed values as normal text."""
    display(Markdown(text))


def n_days(n):
    return f"**{n}** day" if n == 1 else f"**{n}** days"

In [ ]:
# On Google Colab: copy the Excel files from the team's Drive folder.
# Set the folder ID first: import os; os.environ["COSTER_DRIVE_FOLDER"] = "..."
# Google checks that the signed-in account has access to the folder.
try:
    from google.colab import auth
except ImportError:
    auth = None  # running locally, DATA stays as set above

if auth:
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    DRIVE_FOLDER = os.environ["COSTER_DRIVE_FOLDER"]  # the team's Drive folder ID, not in git
    DATA = Path("/content/coster-data")
    DATA.mkdir(exist_ok=True)
    auth.authenticate_user()
    drive = build("drive", "v3")
    found = (
        drive.files()
        .list(
            q=f"'{DRIVE_FOLDER}' in parents and name contains '.xlsx' and trashed = false",
            fields="files(id, name)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        )
        .execute()["files"]
    )
    for f in found:
        target = DATA / f["name"]
        if target.exists():
            continue
        request = drive.files().get_media(fileId=f["id"], supportsAllDrives=True)
        with open(target, "wb") as fh:
            loader, done = MediaIoBaseDownload(fh, request), False
            while not done:
                _, done = loader.next_chunk()
    print(len(found), "Excel files in", DATA)

## 1. The data Coster gave us

Coster sends two kinds of files: **daily** files, sent every working day, and **one-off** files, sent once as a reference.
Below is each file in one line, as Coster describes it, with the short name we use for it in this notebook.

In [ ]:
FILES = {
    "plan": (
        "20261023 - 1 - PU - PRODUZIONE OdL_OdV.xlsx",
        "Foglio1",
        "daily",
        "work orders planned for the next three weeks, with their sales orders",
    ),
    "deposits": (
        "20261023 - 2 - PU - VERSAMENTO UDC.xlsx",
        "Foglio1",
        "daily",
        "load units deposited into the warehouse on the previous day",
    ),
    "releases": (
        "20261023 - 3 - PU - RILASCIO UDC.xlsx",
        "Foglio1",
        "daily",
        "load units released on the previous day",
    ),
    "cells": (
        "20261023 - 4 - PU - SATURAZIONE CELLE MAGAZZINO.xlsx",
        "Foglio1",
        "daily",
        "every cell today, its status and the load unit in it",
    ),
    "cell_summary": (
        "20261023 - 4 - PU - SATURAZIONE CELLE MAGAZZINO.xlsx",
        "Foglio2",
        "daily",
        "number of cells per type and status today",
    ),
    "occupancy": (
        "20261023 - 5 - PU - ANALISI STORICA SATURAZIONE CELLE.xlsx",
        "Foglio1",
        "daily",
        "total, occupied and free cells per day, area and cell type in 2026",
    ),
    "picking": (
        "20261023 - 6 - PU - ANALISI STORICA PICKING.xlsx",
        "Foglio1",
        "daily",
        "pallets used for stamping and load units deleted for production, per day",
    ),
    "inbound": (
        "B01_B02 TO WAREHOUSE PER DAY.xlsx",
        "Foglio1",
        "one-off",
        "approximate number of load units coming in per day and cell type",
    ),
    "machines": (
        "PCS-HOUR.xlsx",
        "Foglio1",
        "one-off",
        "production speed of each machine",
    ),
    "rules": (
        "UDC vs CELLE.xlsx",
        "Foglio1",
        "one-off",
        "which cell types each load unit type may use, and in which order",
    ),
}
pd.DataFrame(
    [(k, f"{v[0]} ({v[1]})", v[2], v[3]) for k, v in FILES.items()],
    columns=["name in this notebook", "file (sheet)", "sent", "what it records (Coster)"],
).set_index("name in this notebook")

**How the files connect.** The files share a few keys: the work order, the sales order, the machine, the load unit type, the cell type and the date.
We check each link in section 3, after cleaning, by counting how many key values of one file are found in the other.

## 2. Cleaning

The files are in Italian and written by an export tool, so they need some work before we can use them. We do five things, in this order:

1. read every value as text, so that codes keep their leading zeros;
2. turn the text `[NULL]` into an empty value and remove extra spaces;
3. give every column an English name, and drop free-text columns we do not need (product, customer and machine descriptions);
4. convert numbers and dates, and count values that fail to convert;
5. replace Italian category values with English ones.

After this section nothing in the tables or charts is in Italian. The table below is the full list of columns we keep:
its original name, the English name, the type and Coster's description of the field.

In [ ]:
# file -> {original column: (English name, type, Coster's description)}
# type: "text", "number", "date" (or "date:<format>"), or None to drop the column
COLUMNS = {
    "plan": {
        "PLANT": ("plant", "text", "production plant"),
        "OdV": ("sales_order", "text", "sales order"),
        "POS": ("sales_order_item", "text", "sales order item"),
        "OdL": ("work_order", "text", "work order"),
        "DT_INIZIO_OdL": ("planned_start", "date:%Y%m%d", "planned work order start date"),
        "DT_FINE_OdL": ("planned_end", "date:%Y%m%d", "planned work order end date"),
        "DT_USCITA_MERCE": ("expected_dispatch", "date:%Y%m%d", "expected goods dispatch date"),
        "ARTICOLO": ("article", "text", "article code"),
        "ARTICOLO_DESC": (None, None, "article description"),
        "QTA": ("planned_qty", "number", "planned production quantity"),
        "MACCHINA": ("machine", "text", "production machine code"),
        "TP_PALLET": ("pallet_type", "text", "required pallet type"),
        "TP_BOX": ("box_type", "text", "required box type"),
        "N_STRATI": ("layers", "number", "pallet configuration: number of layers"),
        "QTA_BOX": ("qty_per_box", "number", "pallet configuration: quantity per box"),
        "N_BOX_STRATO": ("boxes_per_layer", "number", "pallet configuration: boxes per layer"),
        "COD_CLI": ("customer", "text", "customer code"),
        "RAGSOC_CLI": (None, None, "customer name"),
        "COD_DEST": ("destination", "text", "delivery destination code"),
        "RAGSOC_DEST": (None, None, "delivery destination name"),
    },
    "deposits": {
        "PLANT": ("plant", "text", "production plant"),
        "OdL": ("work_order", "text", "work order"),
        "QTA": ("qty", "number", "quantity of pieces produced"),
        "MACCHINA": ("machine", "text", "production machine code"),
        "FLAG_EVASIONE": ("status_flag", "text", "order processing status flag"),
        "DT_REGISTRAZIONE": ("registered_on", "date:%Y%m%d", "load unit registration date"),
    },
    "releases": {
        "PLANT": ("plant", "text", "production plant"),
        "OdL": ("work_order", "text", "work order"),
        "QTA": ("qty", "number", "quantity of pieces produced"),
        "MACCHINA": ("machine", "text", "production machine code"),
        "FLAG_EVASIONE": (
            "status_flag",
            "text",
            "release status flag; A = work order correctly completed",
        ),
        "DT_REGISTRAZIONE": ("registered_on", "date:%Y%m%d", "load unit registration date"),
    },
    "cells": {
        "ID_CELLA": ("cell", "text", "warehouse cell identifier"),
        "ST_ATT": ("availability_code", "number", "cell availability status; 5 = disabled"),
        "ST_ATT_DESC": ("availability", "text", "cell availability status description"),
        "ST_CELLA": ("occupancy_code", "number", "cell occupancy status"),
        "ST_CELLA_DESC": ("occupancy", "text", "cell occupancy status description"),
        "NR_MISS_IN": ("moves_in", "number", "number of inbound pallet missions"),
        "NR_MISS_OUT": ("moves_out", "number", "number of outbound pallet missions"),
        "TP_CELLA": ("cell_type", "text", "cell type code"),
        "TP_CELLA_DESC": ("cell_class", "text", "cell type description"),
        "DT_UM": ("last_change", "date", "last cell modification date"),
        "ID_UDC": ("load_unit", "text", "load unit identifier"),
        "ARTICOLO": ("article", "text", "article code"),
        "TP_UDC": ("load_unit_type", "text", "load unit type"),
        "QTA_BOX": ("qty_per_box", "number", "quantity per box"),
        "QTA": ("qty", "number", "total quantity stored"),
        "ST_QUALITA": ("quality", "text", "quality status: B blocked, L available, S suspended"),
        "ODV": ("sales_order", "text", "associated sales order, where applicable"),
        "POS": ("sales_order_item", "text", "associated sales order item, where applicable"),
        "DT_ESTRAZIONE": ("extracted_on", "date", "data extraction timestamp"),
    },
    "cell_summary": {
        "N_CELLE": ("n_cells", "number", "number of cells in the group"),
        "TP_CELLA": ("cell_type", "text", "cell type code"),
        "TP_CELLA_DESC": ("cell_class", "text", "cell type description"),
        "ST_ATT": ("availability_code", "number", "cell availability status"),
        "ST_ATT_DESC": ("availability", "text", "cell availability status description"),
        "ST_CELLA": ("occupancy_code", "number", "cell occupancy status"),
        "ST_CELLA_DESC": ("occupancy", "text", "cell occupancy status description"),
        "ST_QUALITA": ("quality", "text", "load unit quality status"),
        "DT_ESTRAZIONE": ("extracted_on", "date", "data extraction timestamp"),
    },
    "occupancy": {
        "DT_ANALISI": ("date", "date", "reference date of the analysis"),
        "COD_AREA": ("area", "text", "warehouse area"),
        "TP_CELLA": ("cell_type", "text", "cell type code"),
        "TP_CELLA_DESC": ("cell_class", "text", "cell type description"),
        "TOTALI": ("total", "number", "total number of cells"),
        "OCCUPATE": ("occupied", "number", "number of occupied cells"),
        "LIBERE": ("free", "number", "number of free cells"),
        "PERC_RIEMPIMENTO": ("fill_pct", "number", "occupancy percentage"),
    },
    "picking": {
        "GIORNO": ("date", "date", "reference date of the analysis"),
        "PALLET_STAMP": ("pallets_stamping", "number", "pallets used for stamping"),
        "UDC_CANCELLATE_PROD": (
            "load_units_deleted",
            "number",
            "load units deleted for production purposes",
        ),
    },
    "inbound": {
        "N_UDC": ("load_units", "number", "number of load units received"),
        "TP_CELLA_DESC": ("cell_class", "text", "cell type description"),
        "DT_CREAZIONE": ("date", "date:%d/%m/%Y", "loading date"),
    },
    "machines": {
        "MOULD N": (None, None, "production machine description"),
        "MACCHINA": ("machine", "text", "production machine code"),
        "PCS/HOUR": ("pcs_per_hour", "number", "production rate in pieces per hour"),
    },
    "rules": {
        "TP_UDC": ("load_unit_type", "text", "load unit type code"),
        "TP_UDC_DESC": ("load_unit_desc", "text", "load unit type description"),
        "TP_CELLA": ("cell_type", "text", "cell type code"),
        "TP_CELLA_DESC": ("cell_class", "text", "cell type description"),
        "PRIORITA_UTILIZZO": ("priority", "number", "priority of the match, 1 highest, 5 lowest"),
    },
}
mapping = pd.DataFrame(
    [
        (f, it, en or "(dropped)", kind or "", desc)
        for f, cols in COLUMNS.items()
        for it, (en, kind, desc) in cols.items()
    ],
    columns=["file", "original column", "English name", "type", "description (Coster)"],
).set_index(["file", "original column"])
mapping

The category values are mapped the same way. The cell types and status words are short lists, so we translate them by hand.
For pallet and box types we drop the leading material code and translate the rest.
The load unit type description combines a kind of load unit and a height; we translate the kind and keep the height as a number in millimetres.

In [ ]:
VALUES = {
    "cell_class": {
        "Cella Bassa": "Low",
        "Cella Media": "Medium",
        "Cella Alta": "High",
        "Cella Media (L1000)": "Medium L1000",
        "Cella Alta (L1000)": "High L1000",
    },
    "availability": {"ABILITATA": "enabled", "BLOCCATA": "disabled"},
    "occupancy": {"PIENA": "full", "VUOTA": "empty"},
    "quality": {"L": "available", "B": "blocked", "S": "suspended"},
    "status_flag": {"A": "completed"},
    "pallet_type": {
        "PALLETS 80X120 TRAT. HT ISPM15": "pallet 80x120, heat treated (ISPM15)",
        "PALLETS 100X120TRAT. HT ISPM15": "pallet 100x120, heat treated (ISPM15)",
        "PALLETS 80X120EP.TR. HT ISPM15": "EPAL pallet 80x120, heat treated (ISPM15)",
        "800X1200 CHEP PALLET": "CHEP pallet 800x1200",
    },
    "box_type": {"CART. 395X595X330": "carton 395x595x330"},
}
# kind of load unit, as written before " h." in the type description
LOAD_UNIT_KINDS = {
    "PALLET 1.000 x 1.200": "pallet 1000 x 1200",
    "PALLET 800 x 1.200": "pallet 800 x 1200",
    "PALLET EPAL": "EPAL pallet",
    "PALLET EPAL CHEP": "EPAL CHEP pallet",
    "PALL BOX": "pallbox",
    "PALL BOX AL . RIP.": "foldable pallbox (AL)",
    "CONT. RIPIEGHEVOLE": "foldable container",
    "CONT. FONDELLI": "base-parts container",
    "CONT. FONDELLI CHEP": "base-parts container (CHEP)",
    "CONT. FONDELLI M01208": "base-parts container (M01208)",
    "CONT.INOX": "stainless steel container",
}
ORDER = list(VALUES["cell_class"].values())
pd.DataFrame(
    [(col, it, en) for col, m in VALUES.items() for it, en in m.items()]
    + [("load_unit_desc (kind)", it, en) for it, en in LOAD_UNIT_KINDS.items()],
    columns=["column", "original value", "English value"],
).set_index(["column", "original value"])

Now the cleaning itself. The function below applies the five steps to one sheet and records how many values each step touched,
so that the effect of the cleaning is visible rather than hidden.

In [ ]:
log = []


def clean(key):
    file, sheet, _, _ = FILES[key]
    df = pd.read_excel(DATA / file, sheet_name=sheet, dtype=str)  # step 1
    n = len(df)

    df.columns = df.columns.str.strip()  # step 2
    is_null = df.eq("[NULL]") | df.apply(lambda c: c.str.strip()).eq("[NULL]")
    log.append((key, "2. [NULL] turned into empty", int(is_null.sum().sum()), n))
    df = df.mask(is_null)
    stripped = df.apply(lambda c: c.str.strip())
    log.append(
        (key, "2. extra spaces removed", int(((stripped != df) & df.notna()).sum().sum()), n)
    )
    df = stripped
    if key == "rules":  # the type descriptions also have double spaces inside
        df["TP_UDC_DESC"] = df["TP_UDC_DESC"].str.replace(r"\s+", " ", regex=True)

    cols = COLUMNS[key]  # step 3
    assert set(df.columns) == set(cols), set(df.columns) ^ set(cols)
    dropped = [c for c, (en, _, _) in cols.items() if en is None]
    log.append((key, "3. free-text columns dropped", len(dropped), n))
    df = df.drop(columns=dropped).rename(columns={c: v[0] for c, v in cols.items()})

    failed = 0  # step 4
    for en, kind, _ in cols.values():
        if kind is None or kind == "text":
            continue
        before = df[en].notna()
        if kind == "number":
            df[en] = pd.to_numeric(df[en], errors="coerce")
        else:
            fmt = kind.split(":")[1] if ":" in kind else None
            df[en] = pd.to_datetime(df[en], format=fmt, errors="coerce")
        failed += int((before & df[en].isna()).sum())
    for en in ("load_unit", "load_unit_type", "sales_order", "sales_order_item", "cell_type"):
        if en in df:  # whole numbers that pandas stored as "123.0"
            df[en] = df[en].str.replace(r"\.0$", "", regex=True)
    log.append((key, "4. values that failed to convert", failed, n))

    mapped = 0  # step 5
    for en in ("pallet_type", "box_type"):
        if en in df:  # the leading material code is not needed
            df[en] = df[en].str.replace(r"^\d+ ", "", regex=True)
    for en, values in VALUES.items():
        if en in df:
            unknown = set(df[en].dropna()) - set(values)
            assert not unknown, (key, en, unknown)
            mapped += int(df[en].notna().sum())
            df[en] = df[en].map(values)
    if key == "rules":
        parts = df.pop("load_unit_desc").str.extract(r"^(.*?) h\. ([\d.]+)$")
        assert parts.notna().all().all()
        assert set(parts[0]) <= set(LOAD_UNIT_KINDS), set(parts[0]) - set(LOAD_UNIT_KINDS)
        df["load_unit_kind"] = parts[0].map(LOAD_UNIT_KINDS)
        df["height_mm"] = parts[1].str.replace(".", "", regex=False).astype(int)
        mapped += len(df)
    log.append((key, "5. category values put in English", mapped, n))
    return df


data = {key: clean(key) for key in FILES}
plan, deposits, releases = data["plan"], data["deposits"], data["releases"]
cells, cell_summary, occupancy = data["cells"], data["cell_summary"], data["occupancy"]
picking, inbound, machines, rules = (
    data["picking"],
    data["inbound"],
    data["machines"],
    data["rules"],
)

steps = pd.DataFrame(log, columns=["table", "step", "values touched", "rows"])
steps.pivot_table(index="table", columns="step", values="values touched", sort=False).astype(int)

In [ ]:
total_null = steps.loc[steps.step.str.contains("NULL"), "values touched"].sum()
total_space = steps.loc[steps.step.str.contains("spaces"), "values touched"].sum()
total_failed = steps.loc[steps.step.str.contains("failed"), "values touched"].sum()
say(
    f"The cleaning turned **{total_null:,}** `[NULL]` values into empty values and removed extra "
    f"spaces from **{total_space:,}** values. **{total_failed}** numbers or dates failed to "
    "convert, so every date and number in the files has a valid format. "
    "Every category value found in the files has an English name "
    "(the code stops if one is missing)."
)

**Duplicates.** Each table should have one row per key: one row per work order in the plan, one per cell in the cell table,
one per day, area and cell type in the occupancy history, and so on. We count rows that repeat a key, and rows that repeat entirely.

In [ ]:
KEYS = {
    "plan": ["work_order"],
    "deposits": ["work_order"],
    "releases": ["work_order"],
    "cells": ["cell"],
    "cell_summary": ["cell_type", "availability_code", "occupancy_code", "quality"],
    "occupancy": ["date", "area", "cell_type"],
    "picking": ["date"],
    "inbound": ["date", "cell_class"],
    "machines": ["machine"],
    "rules": ["load_unit_type", "cell_type"],
}
dups = pd.DataFrame(
    {
        "key": {k: " + ".join(v) for k, v in KEYS.items()},
        "rows": {k: len(data[k]) for k in KEYS},
        "rows repeating the key": {k: int(data[k].duplicated(v).sum()) for k, v in KEYS.items()},
        "rows repeated entirely": {k: int(data[k].duplicated().sum()) for k in KEYS},
    }
)
stored_units = cells.load_unit.dropna()
display(dups)
say(
    f"No table has a repeated key or a repeated row. In the cell table, "
    f"{stored_units.duplicated().sum()} load unit numbers appear in more than one cell."
    if dups.iloc[:, 2:].sum().sum() == 0
    else "Some tables repeat a key; see the table above."
)

**Missing values.** The columns below have at least one empty value after cleaning.
An empty value is not always a problem: Coster says some fields are filled only "where applicable".

In [ ]:
missing = pd.DataFrame(
    [
        (k, c, int(df[c].isna().sum()), len(df))
        for k, df in data.items()
        for c in df.columns
        if df[c].isna().any()
    ],
    columns=["table", "column", "empty", "rows"],
).set_index(["table", "column"])
missing["empty %"] = 100 * missing["empty"] / missing["rows"]
missing

In [ ]:
empty_cells = cells.occupancy == "empty"
full = cells.occupancy == "full"
say(
    f"In the cell table, the load unit columns are empty for the **{empty_cells.sum()}** empty "
    f"cells, as expected, and also for **{(full & cells.load_unit.isna()).sum()}** full cells. "
    f"The sales order is filled for **{cells.loc[full, 'sales_order'].notna().mean():.0%}** "
    "of the full cells. "
    f"In the deposits file the status flag is empty on all **{len(deposits)}** rows. "
    f"The picking history has no value for deleted load units on "
    f"**{picking.load_units_deleted.isna().sum()}** days."
)

## 3. Exploration

Before answering questions we look at what each table covers: how many rows, which period, and what the main values look like.

In [ ]:
def period(dates):
    first, last = dates.min(), dates.max()
    return f"{first:%d %b %Y}" if first == last else f"{first:%d %b %Y} to {last:%d %b %Y}"


DATE_COL = {
    "plan": "planned_end",
    "deposits": "registered_on",
    "releases": "registered_on",
    "cells": "extracted_on",
    "cell_summary": "extracted_on",
    "occupancy": "date",
    "picking": "date",
    "inbound": "date",
}
pd.DataFrame(
    {
        "sent": {k: v[2] for k, v in FILES.items()},
        "rows": {k: len(df) for k, df in data.items()},
        "columns": {k: df.shape[1] for k, df in data.items()},
        "period": {k: period(data[k][c]) for k, c in DATE_COL.items()},
        "different dates": {k: str(data[k][c].nunique()) for k, c in DATE_COL.items()},
    }
).fillna("no dates")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))
spans = {k: (data[k][c].min(), data[k][c].max()) for k, c in DATE_COL.items()}
for i, (a, b) in enumerate(spans.values()):
    if a == b:
        ax.plot(a, i, "o", color="tab:blue")
    else:
        ax.plot([a, b], [i, i], lw=6, color="tab:blue", solid_capstyle="butt")
ax.axvline(FILE_DATE, color="grey", ls="--", lw=1)
ax.text(FILE_DATE, -0.6, "file date ", color="grey", fontsize=8, ha="right")
ax.set_yticks(range(len(spans)), list(spans))
ax.invert_yaxis()
ax.set(title="Period covered by each table (one dot = a single day)", xlabel="")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig(OUT / "coverage.png")
plt.show()

In [ ]:
days = pd.date_range(occupancy.date.min(), occupancy.date.max())
missing_days = days.difference(occupancy.date.unique())
say(
    f"Only three tables look back in time: the occupancy history and the picking history "
    f"({period(occupancy.date)}) and the inbound file ({period(inbound.date)}). "
    "The deposits and releases cover one day, the cell table is today, and the plan looks "
    "forward. The occupancy history has no rows on "
    f"{n_days(len(missing_days))} ({', '.join(missing_days.strftime('%d %b')) or 'none'})."
)

**How the tables link.** For each pair of tables that share a key, we count how many key values of the first table are found in the second.

In [ ]:
def link(a, col_a, b, col_b, label):
    va, vb = set(data[a][col_a].dropna()), set(data[b][col_b].dropna())
    return (f"{a} -> {b} ({label})", len(va), len(va & vb))


links = pd.DataFrame(
    [
        link("deposits", "work_order", "plan", "work_order", "work order"),
        link("releases", "work_order", "plan", "work_order", "work order"),
        link("releases", "work_order", "deposits", "work_order", "work order"),
        link("plan", "machine", "machines", "machine", "machine"),
        link("deposits", "machine", "machines", "machine", "machine"),
        link("cells", "load_unit_type", "rules", "load_unit_type", "load unit type"),
        link("cells", "sales_order", "plan", "sales_order", "sales order"),
        link("cells", "article", "plan", "article", "article"),
        link("occupancy", "cell_class", "rules", "cell_class", "cell type"),
        link("inbound", "cell_class", "occupancy", "cell_class", "cell type"),
        link("picking", "date", "occupancy", "date", "date"),
    ],
    columns=["link (key)", "different values", "found in the second table"],
).set_index("link (key)")
links["found %"] = 100 * links.iloc[:, 1] / links.iloc[:, 0]
links

In [ ]:
lk = links["found in the second table"]
say(
    "The reference tables link cleanly: every machine and every load unit type that is used "
    "appears in the machine table and the rules table. The daily tables hardly link at all: "
    f"**{lk['deposits -> plan (work order)']}** of the work orders deposited and "
    f"**{lk['releases -> plan (work order)']}** of those released are in the plan, and "
    f"**{lk['cells -> plan (sales order)']}** of the sales orders stored in the cells are in "
    "the plan. "
    f"The deposits and releases share **{lk['releases -> deposits (work order)']}** work orders. "
    "There is no key that joins one load unit to its work order: the cell table has the load "
    "unit, the article and the sales order, but no work order."
)

**A first look at the main values.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

counts = pd.crosstab(cells.cell_class, cells.occupancy).loc[ORDER]
counts.plot.barh(stacked=True, ax=axes[0], color=["#c7c7c7", "tab:blue"], width=0.7)
axes[0].invert_yaxis()
axes[0].set(title="Cells today", xlabel="cells", ylabel="")
axes[0].legend(fontsize=8)

weekly_plan = plan.groupby(plan.planned_end.dt.to_period("W-SUN").dt.start_time).size()
axes[1].bar(weekly_plan.index.strftime("%d %b"), weekly_plan, width=0.7)
axes[1].set(title="Plan: work orders per week", xlabel="week starting", ylabel="work orders")

rules.groupby("load_unit_type").height_mm.first().value_counts().sort_index().plot.bar(
    ax=axes[2], rot=0
)
axes[2].set(title="Load unit types by height", xlabel="height (mm)", ylabel="types")
plt.tight_layout()
plt.savefig(OUT / "first_look.png")
plt.show()

In [ ]:
say(
    f"Today the warehouse has **{len(cells):,}** cells, of which **{full.sum():,}** are full "
    f"and **{(cells.availability == 'disabled').sum()}** are disabled. "
    f"The plan holds **{len(plan)}** work orders for **{plan.machine.nunique()}** machines. "
    f"The rules cover **{rules.load_unit_type.nunique()}** load unit types in "
    f"**{rules.height_mm.nunique()}** heights."
)

## 4. Analysis

### 4.1 How does occupancy develop over time, per cell class?

The occupancy history gives, for every day, the total, occupied and free cells per area and cell type. We add the two areas together.

In [ ]:
daily = occupancy.groupby(["date", "cell_class"])[["total", "occupied", "free"]].sum()
cap = daily["total"].unstack()[ORDER]
occ = daily["occupied"].unstack()[ORDER]
free = daily["free"].unstack()[ORDER]

fig, axes = plt.subplots(len(ORDER), 1, figsize=(11, 9), sharex=True)
for ax, c in zip(axes, ORDER, strict=True):
    ax.step(cap.index, cap[c], where="post", color="grey", label="capacity")
    ax.step(occ.index, occ[c], where="post", color="tab:blue", label="occupied")
    ax.set_ylim(0, cap[c].max() * 1.1)
    ax.set_ylabel("cells")
    ax.set_title(c, fontsize=10)
axes[0].legend(fontsize=8, ncol=2, loc="lower right")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.suptitle("Occupied cells and capacity per class, 2026", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(OUT / "occupancy.png")
plt.show()

In [ ]:
balance_ok = (occupancy.occupied + occupancy.free == occupancy.total).mean()
changed = cap.diff().fillna(0).ne(0).any(axis=1)
first_day, last_day = cap.index[0], cap.index[-1]
per_day = occupancy.pivot_table(index="date", columns=["area", "cell_type"], values="occupied")
repeat_days = per_day.index[per_day.diff().abs().sum(axis=1) == 0]
display(
    pd.DataFrame({f"{first_day:%d %b}": cap.iloc[0], f"{last_day:%d %b}": cap.iloc[-1]}).T.astype(
        int
    )
)
say(
    f"Occupied plus free equals total on **{balance_ok:.0%}** of the rows. "
    f"The number of cells per class is not fixed: it changed on **{changed.sum()}** days "
    f"({', '.join(cap.index[changed].strftime('%d %b'))}). "
    "So a fill percentage mixes two things, stock and capacity; below we work with cell counts. "
    f"On **{len(repeat_days)}** days the record is identical to the day before in every area and "
    "class, mostly on "
    + " and ".join(d + "s" for d in pd.Series(repeat_days.day_name()).value_counts().index[:2])
    + "."
)

**How often does a class run out of free cells?**

In [ ]:
zero = free == 0
starts = zero & ~zero.shift(1, fill_value=False)
runs = zero.cumsum() - zero.cumsum().where(~zero).ffill().fillna(0)
runout = (
    pd.DataFrame(
        {
            "days with 0 free": zero.sum(),
            "periods with 0 free": starts.sum(),
            "longest period (days)": runs.max(),
            "largest drop in free cells in 1 day": -free.diff().min(),
        }
    )
    .loc[ORDER]
    .astype(int)
)
display(runout)

fig, ax = plt.subplots(figsize=(11, 2.6))
for i, c in enumerate(ORDER):
    ax.scatter(free.index[zero[c]], [i] * zero[c].sum(), marker="|", s=120)
ax.set_yticks(range(len(ORDER)), ORDER)
ax.set(title="Days with 0 free cells, per class", xlabel="2026")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.tight_layout()
plt.savefig(OUT / "zero_free_days.png")
plt.show()

In [ ]:
worst = runout["days with 0 free"].idxmax()
never = [c for c in ORDER if runout.loc[c, "days with 0 free"] == 0]
say(
    f"**Answer.** The history answers this question by day, per class and area, from "
    f"{period(occupancy.date)}. "
    f"**{worst}** ran out most often: **{runout.loc[worst, 'days with 0 free']}** days with no "
    f"free cell, the longest period lasting **{runout.loc[worst, 'longest period (days)']}** days. "
    + (f"{', '.join(never)} never ran out. " if never else "Every class ran out at least once. ")
    + "The history has one record per day and no time of day, so we do not see what happens "
    "within a day. **Status: answered** (daily level)."
)

**Is the cell table a 06:00 snapshot?** A forecast needs a clean starting point.
We compare the full cells in the cell table with the last day of the history, and count the cells whose last change falls on the extraction day itself.

In [ ]:
extraction_day = cells.extracted_on.max()
changed_today = cells.last_change.eq(extraction_day)
enabled_full = full & (cells.availability == "enabled")
timing = (
    pd.DataFrame(
        {
            f"full cells in the cell table ({extraction_day:%d %b})": cells[enabled_full]
            .groupby("cell_class")
            .size(),
            f"occupied in the history ({last_day:%d %b})": occ.iloc[-1],
            "empty cells": cells[empty_cells].groupby("cell_class").size(),
            "empty cells changed on the extraction day": cells[empty_cells & changed_today]
            .groupby("cell_class")
            .size(),
        }
    )
    .reindex(ORDER)
    .fillna(0)
    .astype(int)
)
display(timing)
emp, emp_today = timing["empty cells"], timing["empty cells changed on the extraction day"]
all_changed = [c for c in ORDER if emp[c] > 0 and emp_today[c] == emp[c]]
say(
    f"**{changed_today.sum()}** cells changed on the extraction day itself"
    + (
        f", including every empty cell in **{', '.join(all_changed)}**. "
        "A table taken at 06:00 could only show the first six hours of that day, so it was most "
        "likely taken later in the working day. This is also why it differs from the history. "
        "Until Coster confirms the time (question 8 below), the history stays the target series "
        "and the cell table is not used as a 06:00 starting point for a forecast."
        if all_changed
        else ". Without the extraction time we cannot tell whether the table is a 06:00 snapshot."
    )
)

**Which weekend day is quiet?** On some days the record repeats the day before in every class.
We count, per month, how often the Sunday record and the Monday record repeat the previous day.

In [ ]:
per_day = occ.reindex(pd.date_range(occ.index.min(), occ.index.max()))
moved = per_day.diff().abs().sum(axis=1, min_count=len(ORDER))
quiet = moved.eq(0).astype(float).where(moved.notna())
weekend = pd.DataFrame(
    {"month": quiet.index.to_period("M"), "day": quiet.index.day_name(), "quiet": quiet}
)
monthly = weekend[weekend.day.isin(["Sunday", "Monday"])].pivot_table(
    index="month", columns="day", values="quiet", aggfunc="mean"
)[["Sunday", "Monday"]]
display((100 * monthly).round(0).rename(columns=lambda d: f"{d} record repeats the day before (%)"))
monday_months = [m.strftime("%b") for m in monthly.index[monthly.Monday > monthly.Sunday]]
sunday_months = [m.strftime("%b") for m in monthly.index[monthly.Sunday > monthly.Monday]]
say(
    "The quiet weekend day is not the same all year. "
    "The Monday record repeats Sunday's more often in "
    f"**{', '.join(monday_months) or 'no month'}**. "
    "The Sunday record repeats Saturday's more often in "
    f"**{', '.join(sunday_months) or 'no month'}**. "
    "A forecast with one fixed day-of-week effect mixes the two patterns. "
    "Coster's shift calendar would tell us which days are working days."
)

### 4.2 The life of a load unit: why does it come in, and when and why does it leave?

We follow a load unit through the files: what brings it in, what we know while it is stored, and what takes it out.

**Coming in.** Two files record load units coming in: the deposits (by work order and machine, one day) and the inbound file (counts per day and class, over a longer period).

In [ ]:
inflow = inbound.pivot_table(index="date", columns="cell_class", values="load_units", aggfunc="sum")
inflow = inflow.reindex(pd.date_range(inflow.index.min(), inflow.index.max())).fillna(0)[ORDER]
weekly = inflow.resample("W-MON").sum()

fig, ax = plt.subplots(figsize=(11, 3.5))
bottom = np.zeros(len(weekly))
for c in ORDER:
    ax.bar(weekly.index, weekly[c], bottom=bottom, width=5, label=c)
    bottom += weekly[c].to_numpy()
ax.set(title="Load units coming in per week, by class", xlabel="week", ylabel="load units")
ax.set_ylim(0, weekly.sum(axis=1).max() * 1.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "inflow_weekly.png")
plt.show()

In [ ]:
say(
    f"On {deposits.registered_on.max():%d %b} the deposits file records **{len(deposits)}** work "
    f"orders from **{deposits.machine.nunique()}** machines. Its rows are work orders, not load "
    "units, and it has no load unit number, so the reason we can give for a load unit coming in "
    'is "production of a work order", but we cannot say which load unit came from which order.'
)

**While it is stored.** The cell table shows, for every stored load unit, its quality status, whether it is linked to a sales order, and the date the cell last changed.

In [ ]:
stored = cells[full]
quality = stored.quality.fillna("no code")
linked = np.where(stored.sales_order.notna(), "linked to a sales order", "no sales order")
display(pd.crosstab(quality.rename("quality"), pd.Series(linked, index=stored.index, name="")))

age = (stored.extracted_on - stored.last_change).dt.days
CAP = 91  # weeks up to 13; older cells go in the last bar
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(age.clip(upper=CAP), bins=np.arange(0, CAP + 8, 7))
ax.set(
    title="Full cells by days since the cell last changed",
    xlabel=f"days between last change and extraction (last bar: {CAP} or more)",
    ylabel="cells",
)
plt.tight_layout()
plt.savefig(OUT / "days_since_change.png")
plt.show()

In [ ]:
moves = pd.crosstab(
    [cells.moves_in.rename("moves in"), cells.moves_out.rename("moves out")], cells.occupancy
)
display(moves)
say(
    f"**{(quality != 'available').mean():.0%}** of the stored load units are not available "
    "(blocked, suspended or without a code), and they still take up a cell. "
    f"Half of the full cells have not changed for **{age.median():.0f}** days or more. "
    "The last change date belongs to the cell, not to the load unit, so in general it is not the "
    "date the load unit came in. Section 4.4 shows one family of load units where it nearly is. "
    f"On the extraction day **{(cells.moves_in > 0).sum()}** cells had a move in under way and "
    f"**{(cells.moves_out > 0).sum()}** a move out."
)

**Load units and cells that take capacity but cannot be used.** Blocked or suspended load units still fill a cell, and disabled cells are not counted as capacity. We look at how long they have been in that state.

In [ ]:
held = stored[stored.quality.isin(["blocked", "suspended"])]
held_age = (held.extracted_on - held.last_change).dt.days
dead = (
    pd.DataFrame(
        {
            "blocked or suspended": held.groupby("cell_class").size(),
            "of which unchanged for 30+ days": held[held_age >= 30].groupby("cell_class").size(),
            "capacity": cap.iloc[-1],
        }
    )
    .reindex(ORDER)
    .fillna(0)
    .astype(int)
)
dead["30+ days, % of capacity"] = (
    100 * dead["of which unchanged for 30+ days"] / dead["capacity"]
).round(1)
display(dead)
disabled = cells[cells.availability == "disabled"]
worst_dead = dead["30+ days, % of capacity"].idxmax()
say(
    f"**{len(held)}** load units are blocked or suspended, and "
    f"**{int((held_age >= 30).sum())}** of them have not moved for 30 days or more. "
    f"The largest share is in **{worst_dead}**: "
    f"**{dead.loc[worst_dead, '30+ days, % of capacity']}%** of its capacity. "
    f"The **{len(disabled)}** disabled cells last changed between "
    f"{disabled.last_change.min():%d %b %Y} and {disabled.last_change.max():%d %b %Y}. "
    "Whether these cells and load units count as usable capacity is a question for Coster."
)

**Going out.** Three things can take a load unit out according to the files: a release, a pallet used for stamping, and a load unit deleted for production.
Only the releases name a work order; the picking history gives daily totals for the whole warehouse.

In [ ]:
pick = picking.set_index("date")
by_weekday = pd.DataFrame(
    {
        "load units in": inflow.sum(axis=1).groupby(inflow.index.day_of_week).mean(),
        "pallets used for stamping": pick.pallets_stamping.groupby(pick.index.day_of_week).mean(),
        "load units deleted": pick.load_units_deleted.groupby(pick.index.day_of_week).mean(),
    }
)
by_weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax = by_weekday.plot.bar(figsize=(9, 3.2), rot=0, width=0.8)
ax.set(title="Average per day of the week", xlabel="", ylabel="count per day")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "weekday_profile.png")
plt.show()

**Do the recorded flows explain the change in stock?** If every movement were recorded, the change in occupied cells from one day to the next
would equal what came in minus what went out. We take the deleted load units as what went out (our assumption) and measure how much of the daily change
a straight line on the recorded flows explains (R²).

In [ ]:
def r2(y, X):
    """Share of the variance of y explained by a least-squares line on X."""
    d = pd.concat([y.rename("y"), X], axis=1).dropna()
    A = np.column_stack([np.ones(len(d)), d.drop(columns="y").to_numpy(float)])
    coef, *_ = np.linalg.lstsq(A, d["y"].to_numpy(float), rcond=None)
    rest = d["y"] - A @ coef
    return 1 - (rest**2).sum() / ((d["y"] - d["y"].mean()) ** 2).sum()


occ_days = occ.reindex(pd.date_range(occ.index.min(), occ.index.max()))
change = occ_days.diff().sum(axis=1, min_count=len(ORDER))
flows = pd.DataFrame({"in": inflow.sum(axis=1), "deleted": pick.load_units_deleted}).reindex(
    change.index
)
explained = r2(change, flows)

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(flows["in"] - flows["deleted"], change, s=12, alpha=0.6)
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.set(
    title="Daily change in occupied cells vs recorded flows",
    xlabel="load units in minus load units deleted (per day)",
    ylabel="change in occupied cells (per day)",
)
plt.tight_layout()
plt.savefig(OUT / "balance_scatter.png")
plt.show()

In [ ]:
say(
    f"The recorded flows explain **{explained:.0%}** of the daily change in occupied cells. "
    "The files do not tell us what drives the rest of the change. "
    "\n\n**Answer.** The files give reasons only at an aggregate level: production of work "
    "orders brings load units in; releases, stamping and deletions for production take them "
    "out. None of the files follows a single load unit from the day it enters a cell to the day "
    "it leaves, with a time and a reason. Coster also does not say whether a release means the "
    "load unit leaves its cell. **Status: partly.**"
)

### 4.3 Does the warehouse follow its own plan?

To check this we need, for the same days, what was planned and what actually happened: which work orders were planned and which were deposited, and in what quantity.

In [ ]:
overlap = pd.DataFrame(
    {
        "first date": {
            "plan (planned start)": plan.planned_start.min(),
            "deposits": deposits.registered_on.min(),
            "releases": releases.registered_on.min(),
        },
        "last date": {
            "plan (planned start)": plan.planned_start.max(),
            "deposits": deposits.registered_on.max(),
            "releases": releases.registered_on.max(),
        },
        "work orders": {
            "plan (planned start)": plan.work_order.nunique(),
            "deposits": deposits.work_order.nunique(),
            "releases": releases.work_order.nunique(),
        },
        "of which in the plan": {
            "plan (planned start)": plan.work_order.nunique(),
            "deposits": lk["deposits -> plan (work order)"],
            "releases": lk["releases -> plan (work order)"],
        },
    }
)
overlap

In [ ]:
gap_days = (plan.planned_start.min() - deposits.registered_on.max()).days
say(
    f"**Answer.** The plan starts on {plan.planned_start.min():%d %b}, "
    f"{n_days(gap_days)} after the last day of deposits and releases "
    f"({deposits.registered_on.max():%d %b}). Plan and actual cover different days, and only "
    f"**{lk['deposits -> plan (work order)']}** work order appears in both, so we cannot "
    "compare them yet. "
    "**Status: not in the data yet.** To answer it we need the plans issued for past days, at "
    "least the days covered by the deposits and releases, or several more daily deliveries so "
    "that plan and actual overlap."
)

### 4.4 What does each date mean?

For every date column we put Coster's description next to what the data shows: the period, whether any value has a time of day, and how the date compares with the file date.

In [ ]:
DATES = [
    ("plan", "planned_start"),
    ("plan", "planned_end"),
    ("plan", "expected_dispatch"),
    ("deposits", "registered_on"),
    ("releases", "registered_on"),
    ("cells", "last_change"),
    ("cells", "extracted_on"),
    ("occupancy", "date"),
    ("picking", "date"),
    ("inbound", "date"),
]
english_to_desc = {(f, v[0]): v[2] for f, cols in COLUMNS.items() for v in cols.values()}


def vs_file_date(s):
    lo, hi = (s.min() - FILE_DATE).days, (s.max() - FILE_DATE).days
    return f"{lo:+d} to {hi:+d}" if lo != hi else f"{lo:+d}"


pd.DataFrame(
    [
        (
            f"{t}.{c}",
            english_to_desc[(t, c)],
            period(data[t][c]),
            int((data[t][c] != data[t][c].dt.normalize()).sum()),
            vs_file_date(data[t][c]),
        )
        for t, c in DATES
    ],
    columns=["column", "Coster's description", "period", "values with a time", "days vs file date"],
).set_index("column")

In [ ]:
gap = (plan.expected_dispatch - plan.planned_end).dt.days
bands = pd.cut(
    gap,
    [-999, -1, 0, 3, 7, 999],
    labels=[
        "before end of production",
        "same day",
        "1 to 3 days later",
        "4 to 7 days later",
        "more than 7 days later",
    ],
)
ax = bands.value_counts(sort=False).plot.barh(figsize=(8, 2.8), color="tab:blue")
ax.invert_yaxis()
ax.set(title="Expected dispatch vs planned end of production", xlabel="work orders", ylabel="")
plt.tight_layout()
plt.savefig(OUT / "dispatch_vs_end.png")
plt.show()

In [ ]:
n_times = sum(int((data[t][c] != data[t][c].dt.normalize()).sum()) for t, c in DATES)
start_after_end = (plan.planned_start > plan.planned_end).sum()
changed_after = (cells.last_change > cells.extracted_on).sum()
lag = (FILE_DATE - deposits.registered_on.max()).days
say(
    "The data agrees with Coster's descriptions where we can check them: "
    + (
        "the planned start never comes after the planned end, "
        if start_after_end == 0
        else f"the planned start comes after the planned end on **{start_after_end}** work orders, "
    )
    + f"the deposits and releases are dated {n_days(lag)} "
    'before the file date ("the previous day"), and '
    + (
        "no cell changed after the extraction date. "
        if changed_after == 0
        else f"**{changed_after}** cells changed after the extraction date. "
    )
    + f"One point needs Coster: on **{(gap < 0).sum()}** of **{len(plan)}** work orders the "
    "expected dispatch comes before the planned end of production. "
    + (
        "No date in any file carries a time of day, so no file says at what hour something "
        "happened."
        if n_times == 0
        else f"Only **{n_times}** dates carry a time of day."
    )
    + "\n\n**Answer.** We know what the dates are called and at what level they are recorded, "
    "but there is no put-away or pick date for a single load unit, only the last change of a "
    "cell. **Status: partly.**"
)

**What does the last change date mean for a load unit?** A load unit number looks like a family prefix (the first four digits) followed by a running number. If the running number grows with the last change date, that date is close to the day the load unit was created.

In [ ]:
units = stored.dropna(subset=["load_unit"]).copy()
units["family"] = units.load_unit.str[:4]
units["number"] = pd.to_numeric(units.load_unit.str[4:], errors="coerce")
units["changed"] = units.last_change.astype("int64")
families = units.groupby("family").agg(
    load_units=("load_unit", "size"),
    main_class=("cell_class", lambda s: s.mode().iloc[0]),
)
families["% linked to a sales order"] = (
    100 * units.groupby("family").sales_order.apply(lambda s: s.notna().mean())
).round(0)
families["rank correlation, number vs last change"] = (
    units.groupby("family")[["number", "changed"]]
    .apply(lambda g: g["number"].corr(g["changed"], method="spearman"))
    .round(2)
)
shown = families.copy()
shown["rank correlation, number vs last change"] = shown[
    "rank correlation, number vs last change"
].map("{:.2f}".format)
display(shown)
corr = families["rank correlation, number vs last change"]
large = families.index[families.load_units >= 50]  # ignore families with very few load units
best = corr[large].idxmax()
others = corr[large].drop(best)
say(
    f"Among the families with at least 50 stored load units, **{best}** "
    f"(**{families.loc[best, 'load_units']}** load units, "
    f"**{families.loc[best, '% linked to a sales order']:.0f}%** linked to a sales order) has a "
    f"running number that follows the last change date closely: rank correlation "
    f"**{corr[best]:.2f}**. For these load units the last change date is in practice the day they "
    "came in. For the other large families the correlation is lower ("
    + ", ".join(f"{f}: {corr[f]:.2f}" for f in others.index)
    + "), so there the date can be any later change of the cell."
)

### 4.5 Which load unit type may go into which cell type?

The rules table lists, for each load unit type, the cell classes it may use and the priority of each (1 is the first choice).
We first check how capacity is counted, then group the types by their rule, then check whether the stored load units follow the rules.

In [ ]:
enabled = cells[cells.availability == "enabled"]
cap_check = (
    pd.DataFrame(
        {
            "cells today": cells.groupby("cell_class").size(),
            "disabled": cells[cells.availability == "disabled"].groupby("cell_class").size(),
            "enabled": enabled.groupby("cell_class").size(),
            f"total in history ({last_day:%d %b})": cap.iloc[-1],
        }
    )
    .loc[ORDER]
    .fillna(0)
    .astype(int)
)
display(cap_check)
say(
    f"Each full cell holds one load unit: **{stored.load_unit.nunique()}** different load unit "
    f"numbers in **{stored.load_unit.notna().sum()}** full cells with a number. "
    "The capacity in the history matches the enabled cells in the cell table for "
    f"**{(cap_check.iloc[:, 2] == cap_check.iloc[:, 3]).sum()}** of **{len(ORDER)}** classes, "
    "so disabled cells do not count as capacity."
)

In [ ]:
prio = rules.pivot_table(index="load_unit_type", columns="cell_class", values="priority")[ORDER]
kind = rules.groupby("load_unit_type")[["load_unit_kind", "height_mm"]].first()
kind["wide"] = kind.load_unit_kind.str.contains("1000")
patterns = (
    prio.fillna(0)
    .astype(int)
    .astype(str)
    .replace("0", "-")
    .join(kind)
    .groupby(ORDER)
    .agg(
        types=("height_mm", "size"),
        heights_mm=("height_mm", lambda h: ", ".join(map(str, sorted(set(h))))),
        wide_1000=("wide", "sum"),
        example=("load_unit_kind", "first"),
    )
    .sort_values("types", ascending=False)
    .reset_index()
)
patterns

Each class column is the priority of that class for the group of types, and "-" means the class is not allowed.
`types` counts the load unit types in the group, `heights_mm` lists their heights, and `wide_1000` counts the types that are 1000 mm wide.

In [ ]:
placed = stored.dropna(subset=["load_unit_type"]).merge(
    rules[["load_unit_type", "cell_class", "priority"]],
    on=["load_unit_type", "cell_class"],
    how="left",
)
by_priority = pd.crosstab(placed.cell_class, placed.priority.fillna(0).astype(int))
by_priority.columns = [f"priority {p}" if p else "not allowed" for p in by_priority.columns]
display(by_priority.loc[ORDER])

first_choice = rules[rules.priority == 1].set_index("load_unit_type").cell_class
overflow = placed[placed.priority > 1]
overflow_from = overflow.load_unit_type.map(first_choice)
top = (overflow.cell_class + " cells holding " + overflow_from + "-first types").value_counts()
say(
    f"**{(placed.priority == 1).mean():.0%}** of the stored load units sit in their first-choice "
    f"class and **{placed.priority.isna().sum()}** sit in a class that is not allowed. "
    f"The largest group outside the first choice is {top.index[0]} (**{top.iloc[0]}** load "
    "units). "
    "\n\n**Answer.** The rules are complete for every load unit type in the warehouse, and the "
    "stored load units follow them. The rules depend on the size of the load unit: the lowest "
    "types may use every class, taller types need taller cells, and the 1000-wide types use the "
    "L1000 classes. What the data cannot show is why a load unit went to a later choice, for "
    "example because the first choice was full, since there is no placement time. "
    "**Status: answered** (the rule), **partly** (why a later choice is used)."
)

## 5. Summary and questions for Coster

In [ ]:
pd.DataFrame(
    [
        (
            "1. Occupancy over time, per cell class",
            "answered",
            f"daily, per area and class, {period(occupancy.date)}; capacity changed on "
            f"{changed.sum()} days; {len(repeat_days)} days repeat the day before",
        ),
        (
            "2. Life of a load unit: why in, when and why out",
            "partly",
            f"reasons only as daily totals; recorded flows explain {explained:.0%} of the "
            "daily change; no load unit is followed from entry to exit",
        ),
        (
            "3. Does the warehouse follow its plan",
            "not in the data yet",
            f"plan starts {gap_days} day{'s' * (gap_days != 1)} after the last actual day; "
            f"{lk['deposits -> plan (work order)']} work order in both",
        ),
        (
            "4. What each date means",
            "partly",
            f"descriptions match the data; no time of day; dispatch before end of "
            f"production on {(gap < 0).sum()} work orders",
        ),
        (
            "5. Capacity rules",
            "answered",
            f"{(placed.priority == 1).mean():.0%} in first choice, "
            f"{placed.priority.isna().sum()} not allowed; why a later choice is used is unknown",
        ),
    ],
    columns=["question", "status", "evidence"],
).set_index("question")

**What we need from Coster**

1. A log of movements per load unit: load unit, cell, in or out, date and time, and the reason (production, dispatch, stamping, deletion, internal move).
2. Whether a release means the load unit leaves its cell, and what the status flag in the deposits file means, since it is always empty.
3. The plans issued on past days, at least for the days covered by the deposits and releases, so we can compare plan and actual. Until then, we keep every daily delivery so that plan and actual start to overlap.
4. Why the expected dispatch can come before the planned end of production.
5. The time of day for the dated records, or at least the time each daily file is extracted.
6. Why the history has days missing and days identical to the day before, and why the number of cells per class changed during the year.
7. The expected dispatch date of the load units already in the warehouse, per sales order. Today's plan only holds work orders that start today or later, so it gives a date for very few stored load units. Dispatches cause the large drops that carry most of the forecast error.
8. When the cell table is extracted. It is taken during the working day (section 4.1), so it does not match the 06:00 history. Could it be sent as of 06:00?
9. The shift and plant calendar: which weekend day is worked, per period, and plant holidays (section 4.1).
10. Whether long-blocked or suspended load units and long-disabled cells should count as capacity (section 4.2).
11. What the load unit number prefixes mean, and whether the pallets used for stamping are the new load units of one family.
12. What the deposits and releases count: several machines show more output in one day than 24 hours at their rated speed. And why some work orders are produced without being in the plan.

**Assumptions used**

- The two warehouse areas are added together when we speak of a cell class.
- A load unit deleted for production no longer occupies a cell (section 4.2).
- Dates are local plant dates; no file states a time zone.